In [19]:
import pandas as pd
from psrqpy import QueryATNF

# ---- 1) Template and mapping  --------------------------------------------
template = pd.read_csv('Pulsar_Catalog_Format.csv')['Parameter'].dropna()

# keep only real parameters (no headings like "Foundational Properties")
cols_human = template[~template.str.contains('Properties', na=False)].tolist()

#        Human name           →  ATNF tag
H2A = {'Pulsar ID'              : 'PSRJ',
       'Pulsar Class'           : 'TYPE',
       'Age'                    : 'AGE',
       'Period (P)'             : 'P0',
       'Period Derivative (P˙)' : 'P1',
       'Magnetic Field (B)'     : 'BSURF',
       'Galactic Longitude (l)' : 'GL',
       'Galactic Latitude (b)'  : 'GB',
       'Distance'               : 'DIST',
       'Dispersion Measure (DM)': 'DM',
       'Proper Motion (μl,μb)'  : ['PML','PMB'],   # two columns
       'Kick Velocity'          : 'VTRANS',        # derived
       'Is Binary'              : 'BINARY',
       'Orbital Period (Pb)'    : 'PB',
       'Companion Mass'         : 'M2',
       'Eccentricity (e)'       : 'ECC',
       'Radio Luminosity (L1400)': 'R_LUM14',
      }

# ---- 2) Generic query helper  --------------------------------------------
def make_catalog(outfile, condition=None, psrtype=None, assoc=None, extra_params=None):
    atnf_params = set(sum(([v] if isinstance(v,str) else v for v in H2A.values()), []))
    if extra_params:
        atnf_params |= set(extra_params)

    q = QueryATNF(params=list(atnf_params), condition=condition,
                  psrtype=psrtype, assoc=assoc)
    q.set_derived()                                         # adds AGE, BSURF, VTRANS

    # rename ATNF → human; drop columns not in the template
    df = q.pandas.rename(columns={v:k for k,v in H2A.items() if isinstance(v,str)})
    df = df.reindex(columns=cols_human)                     # right order + NaN fill

    #  ➟ catalogue-specific post-processing
    if outfile.startswith('MSP'):
        df['Pulsar Class'] = df['Pulsar Class'].fillna('MSP')

    df.to_csv(outfile, index=False, float_format='%.6e')
    return df


In [14]:
print(template)

0                               Pulsar ID
1                            Pulsar Class
2                                     Age
4                 Foundational Properties
5                              Period (P)
6                  Period Derivative (P˙)
7                      Magnetic Field (B)
8                 Initial Period (Pinit​)
9                Initial B Field (Binit​)
11     Kinematic & Astrometric Properties
12                 Galactic Longitude (l)
13                  Galactic Latitude (b)
14                               Distance
15                Dispersion Measure (DM)
16                Proper Motion (μl​,μb​)
17                          Kick Velocity
19            Binary & Orbital Properties
20                              Is Binary
21                   Orbital Period (Pb​)
22                   Companion Mass (Mc​)
23                       Eccentricity (e)
25    Emission & Observational Properties
26              Radio Luminosity (L1400​)
27                Single-Pulse Lum

In [21]:
#make_catalog('MSP_catalogue.csv',        condition='P0 < 0.01')
#make_catalog('Normal_PSR_catalogue.csv', psrtype='NRAD')
#make_catalog('RRAT_catalogue.csv',       psrtype='RRAT')
#make_catalog('Binary_MSP_catalogue.csv', condition='P0 < 0.01', psrtype='BINARY',
#             extra_params=['M2'])
make_catalog('GC_PSR_catalogue.csv',     condition='(GL < 1) and (GL > -1) and (GB < 1) and (GB > -1)')
#make_catalog('Halo_PSR_catalogue.csv',   extra_params=['ZZ'])
#make_catalog('GlobularCluster_catalogue.csv', assoc='GC')
#make_catalog('Magellanic_PSR_catalogue.csv',   assoc=['LMC','SMC'])


,Pulsar ID,Pulsar Class,Age,Period (P),Period Derivative (P˙),Magnetic Field (B),Initial Period (Pinit​),Initial B Field (Binit​),Galactic Longitude (l),Galactic Latitude (b),...,Is Binary,Orbital Period (Pb​),Companion Mass (Mc​),Eccentricity (e),Radio Luminosity (L1400​),Single-Pulse Lum. Dist.,Radio Beaming Fraction,Viewing Angle (ζ),Inclination Angle (α),Nulling Fraction (NF)
0,J1745-2758,NaN,1.747244e+09,0.487528,4.420914e-18,4.697922e+10,NaN,NaN,0.847368,0.452347,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,J1746-2829,NaN,2.302174e+04,1.888929,1.300000e-12,5.014521e+13,NaN,NaN,0.450985,0.113105,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,J1746-2849,NaN,1.844497e+06,1.478480,1.270000e-14,4.384899e+12,NaN,NaN,0.134321,-0.029874,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,J1746-2850,NaN,1.270607e+04,1.077101,1.343110e-12,3.848877e+13,NaN,NaN,0.133689,-0.044117,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,J1746-2856,NaN,1.198094e+06,0.945224,1.250000e-14,3.478343e+12,NaN,NaN,0.126180,-0.233344,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,J1747-2802,NaN,1.862462e+07,2.780079,2.365025e-15,2.594756e+12,NaN,NaN,0.970722,0.121393,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,J1747-2809,NaN,5.311513e+03,0.052153,1.555700e-13,2.882385e+12,NaN,NaN,0.869006,0.075936,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,J1750-28,NaN,3.564946e+06,1.300513,5.780000e-15,2.774414e+12,NaN,NaN,0.662858,-0.737239,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
